# Qwen3.5-0.8B + FlyEmbedding-v2 — Qwen-native experiment

This notebook starts directly from **Qwen/Qwen3.5-0.8B**. It does **not** use the trained FlyFFN-v3 model.

The experiment changes only the **input token embedding**. Qwen's transformer/token-mixer stack and original `lm_head` remain unchanged. This isolates whether the Fly/low-rank embedding itself can work without the output-vocabulary distortion that caused empty generations in the previous from-v3 notebook.

Key defaults: rank 896, 16,384 hot-token residual rows, small Fly residual scale, and a generation quality gate.


In [ ]:
#@title 1. Update repository, install dependencies, and preflight
import pathlib, subprocess, sys
REPO_DIR=pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers','accelerate','huggingface_hub','safetensors','ipywidgets','pandas'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)],check=True)
SRC_DIR=REPO_DIR/'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0,str(SRC_DIR))
for p in [REPO_DIR/'scripts'/'run_qwen35_flycore_v2_from_qwen.py', REPO_DIR/'src'/'tinycenn_lm'/'qwen35_flycore_v2.py']:
    subprocess.run([sys.executable,'-m','py_compile',str(p)],check=True)
from tinycenn_lm.qwen35_flycore_v2 import FlyVocabV2Config, FlyVocabCoreV2, FlyEmbeddingV2, assert_qwen35_fly_embedding_v2
print('✓ Qwen-native FlyEmbedding-v2 preflight OK')


In [ ]:
#@title 2. Configuration
BASE_MODEL='Qwen/Qwen3.5-0.8B' #@param {type:'string'}
RUN_MODE='quick' #@param ['quick','strong']
SEQ_LEN=128 #@param {type:'integer'}
BATCH_SIZE=1 #@param {type:'integer'}
VOCAB_LATENT_DIM=896 #@param {type:'integer'}
HOT_TOKEN_COUNT=16384 #@param {type:'integer'}
FLY_NODES=256 #@param {type:'integer'}
GRAPH_STEPS=1 #@param {type:'integer'}
VOCAB_GRAPH_MIX_INIT=0.05 #@param {type:'number'}
MAX_FLY_SCALE=0.05 #@param {type:'number'}
MAX_FINAL_CE_GAP=0.08 #@param {type:'number'}
OUTPUT_DIR=REPO_DIR/'results'/'flycore_v2_from_qwen35_08b'
print('Teacher/source:',BASE_MODEL)
print('FlyFFN-v3 used: NO')
print('Qwen body unchanged: YES')
print('Qwen lm_head unchanged: YES')
print('Only input embedding is replaced by FlyEmbedding-v2')


In [ ]:
#@title 3. Train/evaluate Qwen-native FlyEmbedding-v2 — live output
import subprocess, sys
cmd=[sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_flycore_v2_from_qwen.py'),
     '--base-model',BASE_MODEL,'--run-mode',RUN_MODE,'--seq-len',str(SEQ_LEN),'--batch-size',str(BATCH_SIZE),
     '--vocab-latent-dim',str(VOCAB_LATENT_DIM),'--hot-token-count',str(HOT_TOKEN_COUNT),
     '--fly-nodes',str(FLY_NODES),'--graph-steps',str(GRAPH_STEPS),
     '--vocab-graph-mix-init',str(VOCAB_GRAPH_MIX_INIT),'--max-fly-scale',str(MAX_FLY_SCALE),
     '--max-final-ce-gap',str(MAX_FINAL_CE_GAP),'--output-dir',str(OUTPUT_DIR)]
print('='*100); print(' '.join(cmd)); print('='*100)
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in iter(p.stdout.readline,''): print(line,end='',flush=True)
rc=p.wait(); print('\nFinished, exit code',rc)
if rc: raise subprocess.CalledProcessError(rc,cmd)


In [ ]:
#@title 4. Results
import json, pandas as pd
from IPython.display import display
report=json.loads((OUTPUT_DIR/'report.json').read_text())
hist=pd.read_csv(OUTPUT_DIR/'embedding_training_history.csv')
print('Architecture:',report['architecture'])
print('Base model:',report['base_model'])
print('Qwen body unchanged:',report['qwen_body_unchanged'])
print('Qwen lm_head unchanged:',report['qwen_lm_head_unchanged'])
print('FlyFFN-v3 used:',report['flyffn_v3_used'])
print('\nFactorization:',json.dumps(report['factorization'],indent=2))
print('\nInitial probe:',json.dumps(report['initial_probe'],indent=2))
print('\nFinal probe:',json.dumps(report['final_probe'],indent=2))
print('\nGeneration sanity passed:',report['generation_sanity_passed'])
print('Quality gate passed:',report['quality_gate_passed'])
display(hist.tail(20))
for x in report['generation_sanity']:
    print('\nUSER:',x['prompt']); print('FLY:',x['reply'],'| passed=',x['passed'])


In [ ]:
#@title 5. Reload original Qwen + trained FlyEmbedding checkpoint
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.qwen35_flycore_v2 import FlyVocabV2Config, FlyVocabCoreV2, FlyEmbeddingV2, assert_qwen35_fly_embedding_v2
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype=torch.bfloat16 if device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type=='cuda' else torch.float32)
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL,use_fast=True)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
qwen_model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype if device.type=='cuda' else torch.float32,low_cpu_mem_usage=True).to(device).eval()
fly_model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype if device.type=='cuda' else torch.float32,low_cpu_mem_usage=True).to(device).eval()
ckpt=torch.load(OUTPUT_DIR/'fly_embedding_v2.pt',map_location='cpu',weights_only=True)
cfg=FlyVocabV2Config(**ckpt['config'])
st=ckpt['fly_vocab_core_v2']; adjacency=st['adjacency'].float(); hot_ids=st['hot_token_ids'].long()
ref=fly_model.model.embed_tokens.weight
core=FlyVocabCoreV2(int(fly_model.config.vocab_size),int(fly_model.config.hidden_size),cfg,adjacency,hot_ids,device=ref.device,dtype=ref.dtype)
core.load_state_dict(st,strict=True)
fly_model.fly_vocab_core_v2=core
fly_model.model.embed_tokens=FlyEmbeddingV2(core)
fly_model.config.tie_word_embeddings=False
assert_qwen35_fly_embedding_v2(fly_model)
print('✓ Both models ready. Fly model keeps original Qwen lm_head.')


In [ ]:
#@title 6. Interactive dual chat — original Qwen vs Qwen + FlyEmbedding
import time, html, torch, ipywidgets as widgets
from IPython.display import display, HTML
base_history=[]; fly_history=[]
def _reply(model,history,user_text,max_new=192):
    msgs=history+[{'role':'user','content':user_text}]
    text=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    enc=tokenizer(text,return_tensors='pt').to(device)
    t=time.perf_counter()
    with torch.inference_mode(): out=model.generate(**enc,max_new_tokens=max_new,do_sample=False,use_cache=True,pad_token_id=tokenizer.eos_token_id)
    dt=time.perf_counter()-t
    reply=tokenizer.decode(out[0,enc.input_ids.shape[1]:],skip_special_tokens=True).strip()
    return reply,dt
prompt_box=widgets.Textarea(value='Explain in simple terms why sparse models can save computation.',description='You:',layout=widgets.Layout(width='100%',height='90px'))
ask_button=widgets.Button(description='Ask both models',button_style='success'); reset_button=widgets.Button(description='Reset chat'); outbox=widgets.Output()
def reset_chat(*_):
    global base_history,fly_history
    base_history=[]; fly_history=[]; outbox.clear_output()
def on_ask(_):
    global base_history,fly_history
    q=prompt_box.value.strip()
    if not q: return
    b,bt=_reply(qwen_model,base_history,q); f,ft=_reply(fly_model,fly_history,q)
    base_history += [{'role':'user','content':q},{'role':'assistant','content':b}]
    fly_history += [{'role':'user','content':q},{'role':'assistant','content':f}]
    with outbox:
        display(HTML(f"<table style='width:100%;table-layout:fixed'><tr><th>Original Qwen ({bt:.2f}s)</th><th>FlyEmbedding ({ft:.2f}s)</th></tr><tr><td style='vertical-align:top;white-space:pre-wrap;padding:12px'>{html.escape(b)}</td><td style='vertical-align:top;white-space:pre-wrap;padding:12px'>{html.escape(f)}</td></tr></table>"))
    prompt_box.value=''
ask_button.on_click(on_ask); reset_button.on_click(reset_chat)
display(widgets.VBox([prompt_box,widgets.HBox([ask_button,reset_button]),outbox]))
print('Dual chat ready: original Qwen vs input-only FlyEmbedding-v2.')
